# Import

In [ ]:
import pandas as pd
import json

from curation_tools.curation_tools import (
    CuratedDataset,
    ObsSchema,
    VarSchema,
    Experiment,
    download_file,
    upload_parquet_to_bq
)

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    handlers=[
        logging.FileHandler("curation.log"),
        logging.StreamHandler(),  # keep console output too
    ],
    force=True,
)

# Download data

In [ ]:
noncurated_path = "../non_curated/h5ad/arce_2025.h5ad"
noncurated_dir = "../supplementary/arce_2025/"
download_file(
    url="https://ftp.ncbi.nlm.nih.gov/geo/series/GSE278nnn/GSE278572/suppl/GSE278572%5Fbarcodes.tsv.gz",
    dest_path=noncurated_dir+"GSE278572_barcodes.tsv.gz"
)
download_file(
    url="https://ftp.ncbi.nlm.nih.gov/geo/series/GSE278nnn/GSE278572/suppl/GSE278572%5Ffeatures.tsv.gz",
    dest_path=noncurated_dir+"GSE278572_features.tsv.gz"
)
download_file(
    url="https://ftp.ncbi.nlm.nih.gov/geo/series/GSE278nnn/GSE278572/suppl/GSE278572%5Fmatrix.mtx.gz",
    dest_path=noncurated_dir+"GSE278572_matrix.mtx.gz"
)
download_file(
    url="https://zenodo.org/records/13924126/files/data_tables.zip?download=1",
    dest_path=noncurated_dir+"GSE278572_supplementary_tables.zip",
    unarchive=True
)
download_file(
    url="https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE278572&format=file&file=GSE278572%5Fprotospacer%5Fcalls%5Fper%5Fcell%2Ecsv%2Egz",
    dest_path=noncurated_dir+"GSE278572_protospacer_calls_per_cell.csv.gz",
    unarchive=False
)
!mv ../supplementary/arce_2025/data_tables ../supplementary/arce_2025/supplementary_tables

# Convert to h5ad

Uncomment if running for the first time

In [ ]:
# import scanpy as sc
# adata = sc.read_10x_mtx(
#     noncurated_dir,
#     var_names='gene_symbols',
#     make_unique=False,
#     gex_only=True,
#     prefix="GSE278572_"
# )

# # write to h5ad
# adata.write_h5ad(noncurated_path)

# Initialise the dataset object

In [ ]:
cur_data = CuratedDataset(
    obs_schema=ObsSchema,
    var_schema=VarSchema,
    noncurated_path=noncurated_path
)

cur_data.load_data()

### Add cell barcodes to the obs slot

In [ ]:
cur_data.adata.obs['cell_barcode'] = cur_data.adata.obs.index.astype(str)

print(cur_data.adata.obs[['cell_barcode']].head())

# OBS slot curation

### Show unique perturbations

In [ ]:
# cur_data.rename_columns(slot = 'obs', name_dict = {'sgID_AB': 'perturbation_name'})

### Add guide RNA information

Orignal guideRNA metadata referenced in the paper cannot be unambiguously mapped. Requested updated metadata from the authors - it is contained within the `data_exploration/Perturbseq/supplementary/arce_2025/feature_reference_name_key.csv` folder.

In [ ]:
supp_guides = pd.read_csv("../supplementary/arce_2025/feature_reference_name_key.csv")
supp_guides["guide_sequence"] = supp_guides["sgRNA_long"].str.split("_").str[1]
supp_guides = supp_guides[
    ["id", "target_gene_id", "target_gene_name", "guide_sequence"]
]
supp_guides = supp_guides.replace(
    {"target_gene_id": {"Non-Targeting": "control_nontargeting"},
     "target_gene_name": {"Non-Targeting": "control_nontargeting"}}
)
supp_guides

In [ ]:
# read in the guide RNA spreadsheet
# guides for the essential library are in "ST20" sheet
guide_info_df = pd.read_csv(
    "../supplementary/arce_2025/GSE278572_protospacer_calls_per_cell.csv.gz"
)
guide_info_df = guide_info_df.rename(columns={"feature_call": "perturbation_name"})
guide_info_df = (
    guide_info_df[["cell_barcode", "perturbation_name"]]
    .drop_duplicates()
    .assign(perturbation_name=guide_info_df["perturbation_name"].str.split("|"))
    .explode("perturbation_name")
    .merge(supp_guides, how="left", left_on="perturbation_name", right_on="id")
    .drop(columns=["id"])
    .groupby("cell_barcode", as_index=False)
    .agg({
        "perturbation_name": lambda x: "|".join(x),
        "target_gene_name": lambda x: "|".join(x),
        "target_gene_id": lambda x: "|".join(x),
        "guide_sequence": lambda x: "|".join(x)
    })
)
guide_info_df

In [ ]:
cur_data.adata.obs = cur_data.adata.obs.merge(
    guide_info_df,
    how='left',
    left_on='cell_barcode',
    right_on='cell_barcode'
)
cur_data.adata.obs

### Replace NaNs with control_casonly

In [ ]:
cur_data.adata.obs[cur_data.adata.obs.isna()] = 'control_casonly'
cur_data.adata.obs

### Standardise perturbation targets

In [ ]:
cur_data.standardize_genes(
    slot='obs',
    input_column='target_gene_id',
    input_column_type='ensembl_gene_id',
    multiple_entries=True,
    remove_version=False,
    multiple_entries_sep='|'
    # version_sep='.'
)

In [ ]:
cur_data.adata.obs.head()

### Add `perturbed_target_number` column

In [ ]:
cur_data.count_entries(
    slot='obs',
    input_column='perturbed_target_symbol',
    count_column_name='perturbed_target_number',
    sep='|'
)

### Encode chromosomes as integers

In [ ]:
cur_data.chromosome_encoding()

In [ ]:
cur_data.show_obs(['perturbation_name', 'perturbed_target_chromosome_encoding'])

### Add metadata information from supplementary table 14

In [ ]:
supp_s14 = pd.read_excel(
    "../supplementary/arce_2025/supplementary_tables/S14_metadata_Treg_Teff_perturbseq.xlsx",
    sheet_name="metadata_Treg_total_Teff_total_",
)
supp_s14 = supp_s14[["cell", "donor", "HTO_classification"]]
supp_s14 = supp_s14.rename(
    columns={"cell": "cell_barcode", "donor": "biological_replicate"}
)
supp_s14

In [ ]:
cur_data.adata.obs = cur_data.adata.obs.merge(supp_s14, how="left", on="cell_barcode")

### Drop unmatched cells - these don't have metadata

In [ ]:
cur_data.adata = cur_data.adata[cur_data.adata.obs['biological_replicate'].notna(),:]

In [ ]:
cur_data.adata.obs

### Add metadata

In [ ]:
cur_data.create_columns(
    overwrite=True,
    slot="obs",
    col_dict={
        "dataset_id": cur_data.dataset_id,
        "sample_id": range(1, cur_data.adata.obs.shape[0] + 1),
        # perturbation type
        "perturbation_type_label": "CRISPRi",
        "perturbation_type_id": None,
        "data_modality": "Perturb-seq",
        "significant": None,
        "significance_criteria": None,
        "score_interpretation": None,

        # treatment
        # "treatment_label": None,
        # "treatment_id": None,
        # replicates
        "technical_replicate": None,
        # "biological_replicate": None,
        # model system
        "model_system_label": "primary_cell",
        "model_system_id": None,
        "tissue": "lymphoid tissue",
        "cell_line_label": None,
        "cell_line_id": None,
        # "cell_type_label": None,
        "disease_label": "healthy",
        "disease_id": None,

        "timepoint": "P10DT0H0M0S",
        "species": "Homo sapiens",
        "sex_label": None,
        "sex_id": None,
        "developmental_stage_label": None,
        "developmental_stage_id": None,

        "study_title": "Central control of dynamic gene circuits governs T cell rest and activation",
        "study_uri": "https://doi.org/10.1038/s41586-024-08314-y",
        "study_year": 2025,
        "first_author": "Maya M. Arce",
        "last_author": "Alexander Marson",

        "experiment_title": "Perturb-seq of primary human CD4+ T regulatory and T effector cells under resting and stimulated conditions",
        "experiment_summary": """
            Isolated human Tregs and Teffs from two healthy donors were stimulated with ImmunoCult CD3/CD28/CD2 activator and sequentially transduced with dCas9-KRAB-Zim3 lentivirus (24 hours post-stimulation) and a Perturb-seq guide library (48 hours post-stimulation, MOI 0.3). The library consisted of 28 regulators of IL-2Rα which were subset from Dolcetto library. Cells underwent continuous blasticidin selection starting 48 hours after the first transduction. On day 8, half of the cultures were restimulated using ImmunoCult CD3/CD28/CD2 activator. On day 10, cells were pooled by condition, sorted for live GFP+ expression, and stained with a TotalSeq™-C Universal Cocktail containing hashtag antibodies to distinguish cell types (CD4+ T-effector and T-regulatory) and stimulation conditions (Resting vs. Stimulated). The samples were subsequently processed using the 10x Genomics Chromium Next GEM Single Cell 5' HT v2 platform with Feature Barcode technology and sequenced on an Illumina NovaSeqX.
        """,

        "number_of_perturbed_targets": len(set(cur_data.adata.obs['perturbed_target_coord'])),
        "number_of_perturbed_samples": cur_data.adata.obs.shape[0],

        "library_generation_type_id": "EFO:0022868",
        "library_generation_type_label": "endogenous",

        "library_generation_method_id": None,
        "library_generation_method_label": "dCas9-KRAB-Zim3",

        "enzyme_delivery_method_id": None,
        "enzyme_delivery_method_label": "lentivirus transduction",

        "library_delivery_method_id": None,
        "library_delivery_method_label": "lentivirus transduction",

        "enzyme_integration_state_id": None,
        "enzyme_integration_state_label": "random locus integration",

        "library_integration_state_id": None,
        "library_integration_state_label": "random locus integration",

        "enzyme_expression_control_id": None,
        "enzyme_expression_control_label": "constitutive transgene expression",

        "library_expression_control_id": None,
        "library_expression_control_label": "constitutive transgene expression",

        "library_name": "Human CRISPR Inhibition Pooled Library (Dolcetto)",
        "library_uri": "https://www.addgene.org/pooled-library/broadgpp-human-crispri-dolcetto/",

        "library_format_id": None,
        "library_format_label": "pooled",

        "library_scope_id": None,
        "library_scope_label": "focused",

        "library_perturbation_type_id": None,
        "library_perturbation_type_label": "inhibition",

        "library_manufacturer": "Doench lab",
        "library_lentiviral_generation": "3",
        "library_grnas_per_target": "2",
        "library_total_grnas": str(cur_data.adata.obs['guide_sequence'].str.split('|').explode().nunique()),
        "library_total_variants": None,

        "readout_dimensionality_id": None,
        "readout_dimensionality_label": "high-dimensional assay",

        "readout_type_id": None,
        "readout_type_label": "transcriptomic",

        "readout_technology_id": None,
        "readout_technology_label": "single-cell rna-seq",

        "method_name_id": None,
        "method_name_label": "Perturb-seq",

        "method_uri": None,

        "sequencing_library_kit_id": None,
        "sequencing_library_kit_label": "10x Genomics Chromium Next GEM Single Cell 5-prime HT Kit v2",

        "sequencing_platform_id": None,
        "sequencing_platform_label": "Illumina NovaSeq X",

        "sequencing_strategy_id": None,
        "sequencing_strategy_label": "barcode sequencing",

        "software_counts_id": None,
        "software_counts_label": "CellRanger",

        "software_analysis_id": None,
        "software_analysis_label": "Seurat",

        "reference_genome_id": None,
        "reference_genome_label": "GRCh38",
        
        "license_label": "free to use license",
        "license_id": "SWO:1000061",

        "associated_datasets": json.dumps([
            {
                "dataset_accession": "GSE278572",
                "dataset_uri": "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE278572",
                "dataset_description": "Raw counts; matrix.mtx, features.tsv, barcodes.tsv",
                "dataset_file_name": "GSE278572_*.*",
            }
        ])
    }
)

### Curate tissue information


In [ ]:
cur_data.standardize_ontology(
    input_column='tissue',
    column_type='term_name',
    ontology_type='tissue',
    overwrite=True
)

### Curate cell type information

In [ ]:
cur_data.adata.obs["cell_type"] = cur_data.adata.obs["HTO_classification"].replace(
    {
        "Resting-Teff": "effector CD4-positive, alpha-beta T cell",
        "Stimulated-Teff": "effector CD4-positive, alpha-beta T cell",
        "Resting-Treg": "CD4-positive, CD25-positive, alpha-beta regulatory T cell",
        "Stimulated-Treg": "CD4-positive, CD25-positive, alpha-beta regulatory T cell"
    }
)


In [ ]:
cur_data.standardize_ontology(
    input_column='cell_type',
    column_type='term_name',
    ontology_type='cell_type',
    overwrite=True
)

### Curate cell line information

In [ ]:
# cur_data.standardize_ontology(
#     input_column='cell_line_label',
#     column_type='term_name',
#     ontology_type='cell_line',
#     overwrite=True
# )

### Curate disease information

In [ ]:
cur_data.standardize_ontology(
    input_column='disease_label',
    column_type='term_name',
    ontology_type='disease',
    overwrite=True
)

### Curate treatment information

In [ ]:
cur_data.adata.obs["treatment_label"] = cur_data.adata.obs["HTO_classification"].replace(
    {
        "Resting-Teff": "Untreated Control",
        "Resting-Treg": "Untreated Control",
        "Stimulated-Teff": "anti-CD3 antibody|anti-CD28 antibody|anti-CD2 antibody",
        "Stimulated-Treg": "anti-CD3 antibody|anti-CD28 antibody|anti-CD2 antibody",
    }
)
cur_data.adata.obs["treatment_id"] = cur_data.adata.obs["HTO_classification"].replace(
    {
        "Resting-Teff": "NCIT:C184729",
        "Resting-Treg": "NCIT:C184729",
        "Stimulated-Teff": "EFO:0003317|EFO:0003304|NCIT:C184729",
        "Stimulated-Treg": "EFO:0003317|EFO:0003304|NCIT:C184729",
    }
)

### Match schema column order

In [ ]:
cur_data.match_schema_columns(slot='obs')

### Validate obs metadata

In [ ]:
cur_data.validate_data(slot='obs', verbose=True)

# VAR slot curation

### Standardise genes

In [ ]:
cur_data.show_var()

In [ ]:
cur_data.standardize_genes(
    slot="var",
    input_column="gene_ids",
    input_column_type="ensembl_gene_id",
    remove_version=False,
    multiple_entries=False
)

In [ ]:
cur_data.show_var()


### Replace unmapped gene symbols with original gene symbols

In [ ]:
cur_data.adata.var['gene_symbol'] = cur_data.adata.var['gene_symbol'].fillna(
    cur_data.adata.var['original_index']
)

In [ ]:
cur_data.adata.var

### Validate var metadata

In [ ]:
cur_data.validate_data(slot='var')

# Save the dataset

In [ ]:
cur_data.save_curated_data_h5ad()

In [ ]:
cur_data.save_curated_data_parquet(split_metadata=True, save_metadata_only=True)

# Upload to BigQuery

In [ ]:
upload_parquet_to_bq(
    parquet_path='../curated/parquet/arce_2025_curated_metadata.parquet',
    bq_dataset_id='prj-ext-dev-pertcat-437314.perturb_seq',
    bq_table_name='metadata',
    key_columns=['dataset_id', 'sample_id'],
    verbose=True
)

# Upload to GC Storage

In [ ]:
!gcloud storage cp ../curated/h5ad/arce_2025_curated.h5ad gs://perturbation-catalogue-lake/perturbseq/curated/